In [4]:
import requests
import json
import os
import urllib3
import boto3
import io
import gzip
import logging
from datetime import datetime

In [9]:
s3_client = boto3.client('s3')
today_str = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
bucket_name = "uncc_traffic_data"

In [10]:
s3_bucket = "s3-3601-final-project"
s3_key = f"tomtom/traffic/{today_str}speeds.json.gz"

In [ ]:
api_key = 'AskBella'
#timestamp = datetime.utcnow()


url = f"https://api.tomtom.com/traffic/services/4/flowSegmentData/absolute/10/json"

data_list = []

latitude = [35.307778, 35.313611, 35.311361, 35.309306, 35.306917, 35.293111, 35.294889, 35.299639, 35.287972, 35.301861, 35.311861] # fill in with the found values
longitude = [-80.724583, -80.701944, -80.711222, -80.718056, -80.722639, -80.749361, -80.742111, -80.74575, -80.759472, -80.752333, -80.744]

for i in range(len(latitude)):

        params = {
            'key': api_key,
            'point': f"{latitude[i]},{longitude[i]}",
            'unit': "MPH",
            "openLr": "false"
        }

        response = requests.get(url, params)

        if response.status_code==200:
            data = response.json()
            flow_data = data.get("flowSegmentData", {})

            data_we_want = {
                "timestamp": datetime.now().strftime("%Y-%m-%d_%H-%M-%S"),
                "coordinates": [latitude[i], longitude[i]],
                "current_speed": flow_data.get("currentSpeed"),
                "free_flow_speed": flow_data.get("freeFlowSpeed"),
                "current_delay": flow_data.get("currentDelay"),
                "confidence": flow_data.get("confidence") # for flags of when we shoudl somehow retrain?
            }

            data_list.append(data_we_want)
        else:
            print(f"Error in retrieving data: {response.status_code}")

In [16]:
print(data_list)

[{'timestamp': '2026-09-25_11-38-20', 'coordinates': [35.307778, -80.724583], 'current_speed': 27, 'free_flow_speed': 27, 'current_delay': None, 'confidence': 1}, {'timestamp': '2026-09-25_11-38-20', 'coordinates': [35.313611, -80.701944], 'current_speed': 40, 'free_flow_speed': 40, 'current_delay': None, 'confidence': 1}, {'timestamp': '2026-09-25_11-38-20', 'coordinates': [35.311361, -80.711222], 'current_speed': 22, 'free_flow_speed': 31, 'current_delay': None, 'confidence': 1}, {'timestamp': '2026-09-25_11-38-20', 'coordinates': [35.309306, -80.718056], 'current_speed': 22, 'free_flow_speed': 31, 'current_delay': None, 'confidence': 1}, {'timestamp': '2026-09-25_11-38-21', 'coordinates': [35.306917, -80.722639], 'current_speed': 22, 'free_flow_speed': 31, 'current_delay': None, 'confidence': 1}, {'timestamp': '2026-09-25_11-38-21', 'coordinates': [35.293111, -80.749361], 'current_speed': 29, 'free_flow_speed': 29, 'current_delay': None, 'confidence': 1}, {'timestamp': '2026-09-25_1

In [17]:
compressed = io.BytesIO()
with gzip.GzipFile(fileobj=compressed, mode="wb") as gz:
    gz.write(json.dumps(data_list).encode("utf-8"))
                                    
s3_client.put_object(
    Bucket=s3_bucket,
    Key=s3_key,
    Body=compressed.getvalue(),
    ContentType="application/json",
    ContentEncoding="gzip",
)

{'ResponseMetadata': {'RequestId': 'HB5SKPYTXK87J9AD',
  'HostId': 'cnOE389CRIYX+021lmsLBVBQs1QUU/kiQ4qQI110UUCtOOoTih9gKKGUZMIEmIPlwMLOz2Jw25+aSsqVMEWzDr9aPkva5Rxj',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amz-id-2': 'cnOE389CRIYX+021lmsLBVBQs1QUU/kiQ4qQI110UUCtOOoTih9gKKGUZMIEmIPlwMLOz2Jw25+aSsqVMEWzDr9aPkva5Rxj',
   'x-amz-request-id': 'HB5SKPYTXK87J9AD',
   'date': 'Fri, 25 Sep 2026 15:38:27 GMT',
   'x-amz-server-side-encryption': 'AES256',
   'etag': '"5505e1397d8974fda7700d03ef7fc302"',
   'x-amz-checksum-crc32': '6DU63g==',
   'x-amz-checksum-type': 'FULL_OBJECT',
   'content-length': '0',
   'server': 'AmazonS3'},
  'RetryAttempts': 1},
 'ETag': '"5505e1397d8974fda7700d03ef7fc302"',
 'ChecksumCRC32': '6DU63g==',
 'ChecksumType': 'FULL_OBJECT',
 'ServerSideEncryption': 'AES256'}